# Mixed-layer depth — why `at MLD` comes out blotchy

Every depth subset emits an `_mld` channel, and in almost all of them
that row of the maps looks blotchy in a way the `_sfc`, `_z25m` and
`_mld_mean` rows do not.  This notebook works out why, and what the
alternatives cost.

| | |
|---|---|
| Domain | one 720 × 720 × 51 tile (Section 1) |
| Affects | every `_mld` and `_mld_mean` channel in the DEPTH pipeline |
| Current definition | `calculate_fields_at_depth.mixed_layer_depth` |

## The suspicion, and what is actually going on

The natural suspicion is that `_extract_at_mld` samples a single grid
cell without interpolating, and that interpolation would fix it.  The
first half is true and the second is misleading, and the distinction
decides what to change.

`mixed_layer_depth` returns **the deepest model level Z at which
σ₀ − σ₀(10 m) ≤ 0.03**.  So the MLD it returns is *already* a model
level, not a continuous depth.  Given that, `_extract_at_mld`'s
nearest-k lookup is **exact** — it recovers the level the definition
named, and adding interpolation to the extraction would change
nothing.

The blotchiness comes one step earlier: **MLD is a staircase.**  It can
only take the 51 discrete values in `Z`, and near typical Gulf Stream
mixed layers those are 10–20 m apart.  Two neighbouring columns whose
true mixed layer differs by a metre can therefore land on levels tens
of metres apart, and any field sampled there inherits the jump.  The
map is not noisy — it is quantised.

So the fix, if we want one, is to make **the MLD itself continuous**,
and only then to interpolate the field to it.  Sections 3–5 build that
and compare.


## Section 1 — Setup

In [ ]:
# ---- knobs -------------------------------------------------------------
REGION       = "gulf_stream"          # any region with a 'zoom' anchor
DATE         = "2012-11-09 12:00:00"
LEVELS       = ("sfc", "z25m", "mld", "mld_mean")
ZOOM_HALF_KM = 100.0
# ------------------------------------------------------------------------

import dask
import numpy as np
import xarray as xr

import dbof.preprocessing.calculate_fields as CF
import dbof.preprocessing.calculate_fields_at_depth as CFAD
from dbof.plotting import depth_figures as dfig
from dbof.plotting.field_cmaps import load_field_cmaps
import dbof.utils.native_gradient as NG
from dbof.preprocessing import vertical_helpers as VH
from dbof.tiles import tile_utils
from dbof.tiles.tile_mapping import rect_ij_to_tile

# tile_utils sets the Agg backend on import; restore inline afterwards.
%matplotlib inline
import matplotlib.pyplot as plt

CMAP_CFG, DIVERGING = load_field_cmaps()

S3 = tile_utils._resolve_s3_source(None)
ANCHOR_LON, ANCHOR_LAT = dfig.region_anchor(REGION)
tile = rect_ij_to_tile(
    *tile_utils.latlon_to_rect_ij(ANCHOR_LON, ANCHOR_LAT, S3))
print(f"tile   : idx {tile.tile_idx}, face {tile.face_idx}")

ds_grid = tile_utils._load_grid_for_tile(S3, tile)
ds_raw = tile_utils._load_tracers_for_tile(
    S3, DATE, tile, ["Theta", "Salt", "U", "V", "W"])
ds_merge, xgrid = tile_utils._build_tile_context(ds_raw, ds_grid)

XC, YC = dfig.tile_coords(ds_grid)
LAND = dfig.tile_land_mask(ds_grid)

# Depth coordinate, positive downward, and the layer thicknesses.
Z = np.asarray(VH._get_depth_coord(ds_merge).values, dtype=float)
print(f"levels : {len(Z)}, {Z[0]:.1f} m to {Z[-1]:.1f} m")

# ---------------------------------------------------------------------
# MEMORY -- this one line is why the notebook fits in RAM
# ---------------------------------------------------------------------
# The tile arrives as ONE 720x720x51 chunk, so dask cannot stream: peak
# memory is the whole array, and vertical_stencil_ab holds ~8 full 3D
# intermediates at once, in float64 (the depth coordinate is float64,
# so everything downstream upcasts).  That is ~1.7 GB per field before
# ertel_pv adds ~10 more of its own -- enough to kill the kernel.
#
# Rechunk the HORIZONTAL axes only, ONCE, here.  k must stay whole: the
# mld / mld_mean strategies need entire columns.  Doing it here rather
# than per-section means there is exactly one dataset in the notebook
# and no way to use the wrong one by accident.
# CHUNK = 180 -> 16 blocks of ~13 MB.  Lower it if memory is tight.
# CHUNK = 720 restores production's single-block behaviour: no memory
# benefit, but also no chunk-alignment concerns at all -- use it as the
# escape hatch if anything downstream misbehaves.
CHUNK = 180
ds_merge = ds_merge.chunk({"j": CHUNK, "i": CHUNK})
print(f"chunks : j,i -> {CHUNK} ({(720 // CHUNK) ** 2} blocks), "
      f"k whole")

# Two lazy fields every section below needs.
rho = CF.potential_density(ds_merge)
mld = CFAD.mixed_layer_depth(ds_merge)

## Section 2 — The staircase, shown directly

Two maps.  The MLD in metres, and the **model level index** it
corresponds to.  If the index map is piecewise-constant with sharp
boundaries, the staircase is the mechanism and nothing else needs to
be argued.


In [ ]:
mld = CFAD.mixed_layer_depth(ds_merge)
rho = CF.potential_density(ds_merge)

# The level index each column's MLD lands on.
k_of_mld = xr.apply_ufunc(
    lambda m: np.abs(Z[np.newaxis, :] - m.reshape(-1, 1)).argmin(axis=1)
                .reshape(m.shape).astype("float32"),
    mld, dask="parallelized", output_dtypes=[np.float32],
)

vals = dfig.pack_tile_levels(
    dict(zip(["mixed_layer_depth", "k_of_mld"],
             dask.compute(mld, k_of_mld))),
    XC, YC, edge_margin=0, land_mask=LAND, levels=("sfc",),
    verbose=False)

print("level spacing where the MLD lands:")
kk = vals["k_of_mld"]["sfc"][2]
for q in (5, 25, 50, 75, 95):
    k = int(np.nanpercentile(kk, q))
    print(f"  p{q:<3d}  k={k:2d}  z={Z[k]:7.1f} m   "
          f"gap to next level {Z[min(k + 1, len(Z) - 1)] - Z[k]:5.1f} m")
print(f"\ndistinct levels used across the tile: "
      f"{int(np.nanmax(kk) - np.nanmin(kk)) + 1}")

In [ ]:
dfig.depth_map_grid(
    ["mixed_layer_depth", "k_of_mld"], vals, CMAP_CFG,
    region=REGION, levels=("sfc",),
    row_labels=("whole tile", f"{2 * ZOOM_HALF_KM:.0f} km zoom"),
    diverging_cmaps=DIVERGING, zoom_half_km=ZOOM_HALF_KM,
    suptitle=("Figure 1 — MLD in metres, and the model level it snaps "
              "to.  The second panel is the staircase."))
plt.show()

## Section 3 — A continuous MLD

Same 0.03 kg m⁻³ criterion, but instead of returning the last level
that satisfies it, interpolate **linearly in z** between the bracketing
levels to find where the threshold is crossed exactly.  This is the
smallest possible change to the definition: same threshold, same
reference depth, continuous output.


In [ ]:
def mld_interpolated(ds_merge, threshold=0.03, ref_depth_m=10.0):
    """Continuous MLD: linear interpolation to the exact crossing.

    Same criterion as ``calculate_fields_at_depth.mixed_layer_depth``
    (sigma0 exceeding a reference value by *threshold*), but the depth
    is interpolated between the two bracketing model levels instead of
    being snapped to the deeper one.

    Inputs
    ------
    ds_merge : xr.Dataset
        Merged tile dataset.
    threshold : float
        Density criterion, kg m-3.
    ref_depth_m : float
        Reference depth for sigma0, m.

    Outputs
    -------
    xr.DataArray
        2D MLD in metres, positive downward, dask-backed.

    Generated by LH and Claude.
    """
    sigma0 = dfig.align_chunks(CF.potential_density_anomaly(ds_merge))
    zdim = VH._get_vertical_dim(sigma0)
    k_ref = int(np.abs(Z - ref_depth_m).argmin())

    def _interp(sig):
        # sig: (..., nk) -- apply_ufunc puts the vertical axis last.
        flat = sig.reshape(-1, sig.shape[-1]).astype(np.float64)
        excess = flat - (flat[:, k_ref][:, None] + threshold)
        out = np.full(flat.shape[0], Z[-1], dtype=np.float64)
        for n in range(flat.shape[0]):
            e = excess[n]
            idx = np.nonzero(e > 0)[0]
            idx = idx[idx > k_ref]
            if idx.size == 0:
                continue
            k = idx[0]
            e0, e1 = e[k - 1], e[k]
            # Linear crossing of excess == 0 between k-1 and k.
            out[n] = (Z[k - 1] + (Z[k] - Z[k - 1]) * (-e0) / (e1 - e0)
                      if e1 != e0 else Z[k])
        return out.reshape(sig.shape[:-1]).astype(np.float32)

    return xr.apply_ufunc(
        _interp, sigma0,
        input_core_dims=[[zdim]], dask="parallelized",
        output_dtypes=[np.float32],
    )


mld_i = mld_interpolated(ds_merge)
mld_di = CFAD.mixed_layer_depth_DI(ds_merge)

mlds = dict(zip(["mld_threshold", "mld_interp", "mld_DI"],
                dask.compute(mld, mld_i, mld_di)))
mv = dfig.pack_tile_levels(mlds, XC, YC, edge_margin=0,
                           land_mask=LAND, levels=("sfc",),
                           verbose=False)

print(f"{'definition':<16}{'median':>10}{'p5':>9}{'p95':>9}"
      f"{'distinct values':>18}")
print("-" * 62)
for nm in ("mld_threshold", "mld_interp", "mld_DI"):
    a = mv[nm]["sfc"][2]
    f = a[np.isfinite(a)]
    print(f"{nm:<16}{np.median(f):>10.1f}{np.percentile(f, 5):>9.1f}"
          f"{np.percentile(f, 95):>9.1f}{len(np.unique(f)):>18d}")
print("")
print("'distinct values' is the staircase, quantified: the threshold")
print("definition can only return model-level depths.")

In [ ]:
dfig.depth_map_grid(
    ["mld_threshold", "mld_interp", "mld_DI"], mv, CMAP_CFG,
    region=REGION, levels=("sfc",),
    row_labels=("whole tile", f"{2 * ZOOM_HALF_KM:.0f} km zoom"),
    diverging_cmaps=DIVERGING, zoom_half_km=ZOOM_HALF_KM,
    suptitle="Figure 2 — three MLD definitions on the same tile")
plt.show()

dfig.depth_pdf_grid(
    ["mld_threshold", "mld_interp", "mld_DI"], mv, CMAP_CFG,
    levels=("sfc",), row_labels=("whole tile",),
    suptitle=("Figure 3 — MLD distributions.  The threshold definition "
              "should show comb teeth at the model levels."))
plt.show()

Figure 3 is the test.  If `mld_threshold` shows **comb teeth** — spikes
at the model-level depths with gaps between — while `mld_interp` is
smooth, the staircase is confirmed and quantified.


### What DI stands for, and why it looks smoothest

**DI = Depth Integration.**  `mixed_layer_depth_DI` defines the MLD as
the **N²-weighted mean depth** over a fixed integration depth — a
centroid of the stratification profile, not a threshold crossing.

That is why it is the smoothest of the three, and the reason is worth
being suspicious of: **it is smooth because it is an integral.**  Every
column averages over the whole profile, so single-level structure is
smoothed away by construction.  That is not evidence that it is a
better estimate of the mixed-layer depth — it is a different quantity
with different units of meaning, and Figure 2 shows its *values* sit
well below the threshold definitions.

So do not pick DI because the map looks nicer.  Pick it, if at all,
because an N²-weighted centroid is what a downstream field actually
wants.  `Fr` and `KE` divide by MLD as a length scale, and a centroid
may genuinely suit them better than a threshold crossing — but that is
a physics argument to make explicitly, not a smoothness contest.


## Section 4 — Order of operations

This is the question that actually decides what to change.  To get a
property "at the MLD" there are three orders available, and they are
not equivalent:

| | Order | What it means |
|---|---|---|
| **A** | compute in 3D → **snap** to nearest level | production today |
| **B** | compute in 3D → **interpolate** in z to a continuous MLD | keeps the along-level meaning, removes the staircase |
| **C** | interpolate the **inputs** to the MLD → compute | computes along the MLD *surface*, not along a level |

**A and B differ only in the sampling.**  Both compute the property on
level surfaces and then ask "what is its value at the mixed-layer
depth".  B removes the staircase.

**C is a different physical quantity, and for horizontal gradients it
is dangerous.**  A gradient taken along the tilted MLD surface is

    ∇b|_MLD-surface  =  ∇b|_z  +  (∂b/∂z)·∇(MLD)

so it picks up the slope of the MLD field itself.  Since the production
MLD is a *staircase*, ∇(MLD) is a field of spikes at the level
boundaries — C would inject those straight into `gradb2`.  Section 5
shows this happening.

The pipeline uses **A** everywhere.  This section measures what B would
buy, across the six field types.


In [ ]:
# ---------------------------------------------------------------------
# WHAT THE PIPELINE ACTUALLY DIFFERENTIATES IN z
# ---------------------------------------------------------------------
# Every _vertical_derivative call site in src/dbof, in full:
#
#   calculate_fields_at_depth.py:153   _vertical_derivative(rho)  -> N2
#   calculate_fields_at_depth.py:313   _vertical_derivative(U)    -> u_z
#   calculate_fields_at_depth.py:314   _vertical_derivative(V)    -> v_z
#   calculate_fields_at_depth.py:574   _vertical_derivative(b)    -> b_z
#
# That is the complete list.  Three distinct inputs: potential density,
# U and V.  (b = g*sigma0/rho0 is rho times a constant, so d(b)/dz and
# d(rho)/dz have IDENTICAL sign structure -- a useful self-check.)
#
# We NEVER take the vertical derivative of a horizontal gradient, of a
# Jacobian, of N2, or of ertel_pv.  So for those fields the stencil
# diagnostic is COUNTERFACTUAL -- it answers a question the pipeline
# never asks.  They are still worth plotting, but for a different
# reason: see the note under Figure 2.
# ds_merge was already rechunked in Section 1, so every graph built
# here streams.  jac is shared by two of the probe fields.
jac = CF.compute_velocity_jacobian(ds_merge, xgrid)

# Group A -- the pipeline really does take d/dz of these.
DIFFERENTIATED = {
    "rho": rho,
    "U": ds_merge["U"],
    "V": ds_merge["V"],
}

# Group B -- never differentiated in z.  Here the same diagnostic reads
# as a VERTICAL ROUGHNESS probe: how much fine structure the field has
# from one level to the next.  That is not a stencil error, but it IS
# what governs how badly a field blotches when sampled at a
# staircase-quantised MLD (see mixed_layer_depth.ipynb).
ROUGHNESS_PROBE = {
    "Theta": ds_merge["Theta"],
    "gradb2": CF.grad_b2(ds_merge, xgrid),
    "relative_vorticity": CF.relative_vorticity(
        ds_merge, xgrid, jacobian=jac),
    "N2": CFAD.buoyancy_frequency_squared(ds_merge),
    "ertel_pv": CFAD.ertel_pv_terms(ds_merge, xgrid)["ertel_pv"],
}

ZOO = {**DIFFERENTIATED, **ROUGHNESS_PROBE}
ZOO_KIND = {
    "rho": "DIFFERENTIATED -> N2",
    "U": "DIFFERENTIATED -> u_z",
    "V": "DIFFERENTIATED -> v_z",
    "Theta": "probe: raw tracer",
    "gradb2": "probe: horizontal gradient",
    "relative_vorticity": "probe: horizontal Jacobian",
    "N2": "probe: already a vertical gradient",
    "ertel_pv": "probe: vertical x horizontal product",
}
# Trim this if the kernel still dies -- ertel_pv is by far the heaviest.
ZOO_FIELDS = list(ZOO)

print(f"differentiated by the pipeline : {list(DIFFERENTIATED)}")
print(f"roughness probes only          : {list(ROUGHNESS_PROBE)}")
print(f"running                        : {ZOO_FIELDS}")

In [ ]:
mld_i = mld_interpolated(ds_merge)


def field_at_depth_interp(field3d, depth2d):
    """Sample a 3D field at a continuous depth, linear in z.

    The counterpart of ``VH._extract_at_mld`` for a continuous MLD:
    interpolates between the bracketing levels instead of snapping to
    the nearer one.

    Inputs
    ------
    field3d : xr.DataArray
        Lazy 3D field on tracer levels.
    depth2d : xr.DataArray
        2D target depth, m, positive downward.

    Outputs
    -------
    xr.DataArray
        2D field, dask-backed.

    Generated by LH and Claude.
    """
    zdim = VH._get_vertical_dim(field3d)

    def _interp(f, d):
        flat = f.reshape(-1, f.shape[-1]).astype(np.float64)
        dd = d.ravel().astype(np.float64)
        k = np.clip(np.searchsorted(Z, dd), 1, len(Z) - 1)
        z0, z1 = Z[k - 1], Z[k]
        w = np.where(z1 > z0, (dd - z0) / (z1 - z0), 0.0)
        rows = np.arange(flat.shape[0])
        out = (1 - w) * flat[rows, k - 1] + w * flat[rows, k]
        return out.reshape(d.shape).astype(np.float32)

    return xr.apply_ufunc(
        _interp, field3d, depth2d,
        input_core_dims=[[zdim], []], dask="parallelized",
        output_dtypes=[np.float32],
    )


# A vs B for every field in the zoo.
# align_chunks is not optional here.  These two calls index a 3D field
# and a 2D depth together inside one apply_ufunc, so their horizontal
# chunks must match -- and a field that came through xgcm's diff/interp
# is often chunked differently from the MLD even though both started
# from ds_merge.  Without this the failure is an IndexError deep inside
# the ufunc ("could not be broadcast together with shapes ...").
pairs = {}
for nm, fld in ZOO.items():
    f_a, mld_a = dfig.align_chunks(fld, mld)
    pairs[f"{nm}__A_snap"] = VH._extract_at_mld(f_a, mld_a, ds_merge)
    f_b, mldi_a = dfig.align_chunks(fld, mld_i)
    pairs[f"{nm}__B_interp"] = field_at_depth_interp(f_b, mldi_a)
vals = dict(zip(pairs, dask.compute(*pairs.values(), retries=10)))
print("computed:", len(vals), "fields")

In [ ]:
def roughness(a):
    """Median absolute discrete Laplacian, ignoring NaNs.

    A staircase makes large cell-to-cell jumps at the level boundaries
    and so scores high; smooth physical structure scores low.
    """
    lap = (a[2:, 1:-1] + a[:-2, 1:-1] + a[1:-1, 2:] + a[1:-1, :-2]
           - 4 * a[1:-1, 1:-1])
    return np.nanmedian(np.abs(lap))


def _clean(v):
    a = np.squeeze(np.asarray(getattr(v, "values", v))).astype("float32")
    return np.where(LAND, np.nan, a)


print(f"{'field':<20}{'kind':<36}{'A snap':>12}{'B interp':>12}"
      f"{'smoother by':>13}")
print("-" * 93)
for nm in ZOO:
    ra = roughness(_clean(vals[f"{nm}__A_snap"]))
    rb = roughness(_clean(vals[f"{nm}__B_interp"]))
    print(f"{nm:<20}{ZOO_KIND[nm]:<36}{ra:>12.3e}{rb:>12.3e}"
          f"{ra / max(rb, 1e-30):>12.1f}x")
print("")
print("'smoother by' > 1 means interpolating removed roughness that the")
print("snap-to-level sampling had introduced.  Watch whether the gain is")
print("bigger for the already-differentiated fields.")

In [ ]:
# Maps for two contrasting cases: a raw tracer and a derived field.
for nm in ("Theta", "ertel_pv"):
    mv2 = dfig.pack_tile_levels(
        {"A_snap": vals[f"{nm}__A_snap"],
         "B_interp": vals[f"{nm}__B_interp"]},
        XC, YC, edge_margin=3, land_mask=LAND, levels=("sfc",),
        verbose=False)
    dfig.depth_map_grid(
        ["A_snap", "B_interp"], mv2, CMAP_CFG, region=REGION,
        levels=("sfc",),
        row_labels=("whole tile", f"{2 * ZOOM_HALF_KM:.0f} km zoom"),
        diverging_cmaps=DIVERGING, zoom_half_km=ZOOM_HALF_KM,
        suptitle=(f"Figure 3 — {nm} at the MLD: snapped to a model "
                  f"level (A) vs interpolated (B)"))
    plt.show()

## Section 5 — Why order C is a trap for horizontal gradients

Order C — interpolate the inputs to the MLD, then take the horizontal
gradient — sounds like the natural way to "get the gradient at the
MLD".  It is not, and this section shows why in one figure.

Taking ∇ along the MLD surface adds a `(∂b/∂z)·∇(MLD)` term.  With a
staircase MLD, `∇(MLD)` is zero almost everywhere and enormous at the
level boundaries, so C turns those boundaries into artificial fronts.


In [ ]:
b = CF.buoyancy_of_field(ds_merge)

# A: gradient computed on level surfaces, then sampled at the MLD.
_g3, _m = dfig.align_chunks(CF.grad_b2(ds_merge, xgrid), mld)
gradb2_A = VH._extract_at_mld(_g3, _m, ds_merge)

# C: buoyancy sampled at the MLD first, then a horizontal gradient of
# that 2D field.  Uses the same staggered machinery, so the only
# difference is the order.
_b3, _m = dfig.align_chunks(b, mld)
b_at_mld = VH._extract_at_mld(_b3, _m, ds_merge)
ds_c = ds_merge.assign(_b_mld=b_at_mld)
bx_c, by_c = NG.calculate_native_gradient_tracer(
    ds_c["_b_mld"], ds_merge, grid=xgrid)
gradb2_C = bx_c ** 2 + by_c ** 2

gv = dfig.pack_tile_levels(
    dict(zip(["A_grad_then_sample", "C_sample_then_grad"],
             dask.compute(gradb2_A, gradb2_C))),
    XC, YC, edge_margin=3, land_mask=LAND, levels=("sfc",),
    verbose=False)

dfig.depth_map_grid(
    ["A_grad_then_sample", "C_sample_then_grad"], gv, CMAP_CFG,
    region=REGION, levels=("sfc",),
    row_labels=("whole tile", f"{2 * ZOOM_HALF_KM:.0f} km zoom"),
    diverging_cmaps=DIVERGING, log_scale_channels={"A_grad_then_sample",
                                                   "C_sample_then_grad"},
    zoom_half_km=ZOOM_HALF_KM,
    suptitle=("Figure 4 — |∇b|² at the MLD: gradient-then-sample "
              "(production) vs sample-then-gradient"))
plt.show()

ra = roughness(gv["A_grad_then_sample"]["sfc"][2])
rc = roughness(gv["C_sample_then_grad"]["sfc"][2])
print(f"roughness  A (production) {ra:.3e}   C {rc:.3e}   "
      f"C is {rc / max(ra, 1e-30):.1f}x rougher")
print("")
print("If C is dramatically rougher and its structure traces the MLD")
print("level boundaries rather than the fronts, the extra term is the")
print("staircase gradient and order C should never be used.")

## Section 6 — Decision

Options, in increasing order of disruption:

1. **Change nothing.**  Document that `_mld` is quantised to model
   levels and read it accordingly.  Free, and honest.
2. **Continuous MLD + interpolated extraction** (order B throughout).
   Internally consistent, and the only option that actually smooths the
   `_mld` channels.  Section 4 measures what it buys per field type;
   the interesting question is whether the gain is concentrated in the
   already-differentiated fields, because those are the ones whose
   `_mld` rows look worst.
3. **Switch the MLD definition** (DI or an N²-max criterion).  A
   different quantity, not a smoothing fix — see the note above
   Section 4.

Whatever is chosen, order **C** should be ruled out explicitly for any
horizontal gradient, and `docs/Fields.md` should say which order the
`_mld` and `_mld_mean` channels mean.  Right now it does not, and the
difference is not obvious from the channel name.

Note that B is not free of artifacts either: interpolating in z across
a sharp pycnocline smooths real structure along with the staircase.
The question is which error we would rather have, and that is LH's
call, not something these numbers settle.

---

### Cross-references

- **The vertical stencil**, a different mechanism with similar
  symptoms — `vertical_gradients.ipynb`.
- **N² and the MLD channels themselves** —
  `depth_fields/stratification.ipynb`.
- **The fields whose `_mld` rows look worst** —
  `depth_fields/frontal_structure.ipynb`,
  `depth_fields/mixing_parameters.ipynb`,
  `depth_fields/ertel_pv.ipynb`.
